<div align="center">
    <h2><b>Architecture Design Document (ADD)</b></h2>
    <h1>Обробка великих обсягів даних у системах (Big Data & Streaming)</h1>
    <h3>Муніципальна система «ctOSmartCity»</h3>
    <br>

**Метадані документа:**

| **Тип документа:** | High-Level Design (HLD) & Communication Strategy |
| :--- | :--- |
| **Статус:** | Final / Release |
| **Рівень конфіденційності:** | Internal Use Only (Privacy by Design) |
| **Підрядник:** | WatchDog Analytics (Blume Corporation) |
| **Версія архітектури:** | 1.0.0 |
| **Дата створення:** | Березень 2026 р. |

<br>

**Автор проєкту:**

| **Ім'я:** | Олег Гаценко |
| :--- | :--- |
| **Роль:** | Data Architect / Студент магістратури |
| **Організація:** | NeoVersity (Master's in AI & Machine Learning) |

</div>

---

### **Executive Summary (Резюме архітектури):**

Цей документ описує високорівневу архітектуру для платформи розумного міста **ctOSmartCity**. Система щосекунди агрегує петабайти подій від тисяч IoT-сенсорів (світлофори, рівень CO₂, камери трафіку, паркінги) в умовах суворих вимог до конфіденційності даних (Zero Trust). 

Оскільки платформа має одночасно забезпечувати **миттєве реагування** (автоматичний виклик екстрених служб під час ДТП) та **глибоку історичну аналітику** (місячні звіти для мерії), базовим патерном проєктування було обрано **Лямбда-архітектуру (Lambda Architecture)**.

**Ключові архітектурні рішення:**
* **Ingestion Layer:** Асинхронний збір подій через `Apache Kafka` (єдине джерело правди та надійний буфер, що згладжує пікові навантаження).
* **Speed Layer (Гарячий шлях):** `Apache Flink` для Complex Event Processing (CEP) та виявлення аномалій у реальному часі (з гарантією доставки Exactly-Once).
* **Batch Layer (Холодний шлях):** Зберігання сирих даних у Data Lake (`HDFS` / `AWS S3`) у форматі Parquet та їх пакетна обробка через `Apache Spark`.
* **Serving Layer:** Швидка NoSQL база `Apache Cassandra` для мілісекундної віддачі метрик кінцевим споживачам.

### **Контекстна діаграма потоків (Level 0):**

```mermaid
graph LR
    %% Styling
    classDef source fill:#2b2b2b,stroke:#00ffcc,stroke-width:2px,color:#fff
    classDef stream fill:#1a1a1a,stroke:#ff3366,stroke-width:2px,color:#fff
    classDef batch fill:#1a1a1a,stroke:#3399ff,stroke-width:2px,color:#fff
    classDef sink fill:#2b2b2b,stroke:#ffcc00,stroke-width:2px,color:#fff
    classDef user fill:#1a1a1a,stroke:#ffffff,stroke-width:2px,color:#fff

    IoT(("📡 IoT<br>Сенсори")):::source
    Kafka{{"📥 Ingestion<br>(Apache Kafka)"}}:::source

    Flink["⚡ Speed Layer<br>(Apache Flink)"]:::stream
    Spark["🧊 Batch Layer<br>(HDFS + Spark)"]:::batch

    DB[("📊 Serving<br>(Cassandra)")]:::sink
    Alerts[["🚨 Екстрені<br>Служби"]]:::sink

    App(("📱 Мобільний<br>застосунок")):::user
    Gov(("🏛️ Dashboard<br>Мерії")):::user

    IoT -- "Мільйони подій/сек" --> Kafka
    Kafka -- "Безперервний стрім" --> Flink
    Kafka -- "Скидання логів" --> Spark

    Flink -- "Миттєвий Alert (Exactly-Once)" --> Alerts
    Flink -- "Real-time агрегації" --> DB
    Spark -- "Важкі історичні звіти" --> DB

    DB -. "API Gateway (Read)" .-> App
    DB -. "API Gateway (Read)" .-> Gov
```

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **1. Архітектура системи (Лямбда-архітектура)**

Для реалізації платформи **ctOSmartCity** обрано патерн **Lambda Architecture**, який чітко розмежовує потокову (Stream) та пакетну (Batch) обробку. Це гарантує, що важкі аналітичні запити (формування місячних звітів) ніколи не вплинуть на швидкість реакції системи на критичні події (аварії).

### 1.1. Основні компоненти та їх ролі

- **Джерела подій (IoT Devices):** Тисячі міських сенсорів (камери трафіку, датчики CO₂, розумні паркінги). Дані передаються через легковісний протокол **MQTT** на IoT Gateway.
- **Транспорт (Ingestion Layer) — Apache Kafka:**
    -  *Роль:* Центральна "нервова система" міста, єдине джерело правди (Single Source of Truth) та розподілений буфер.
    - *Механіка:* Сирі події (у форматі JSON) записуються у відповідні топіки. Kafka згладжує будь-які пікові навантаження та дозволяє паралельно читати одні й ті самі дані різним підсистемам.
- **Потокова обробка (Speed Layer) — Apache Flink:**
    - *Роль:* Complex Event Processing (CEP) та аналіз у реальному часі.
    - *Механіка:* Безперервно читає стрім із Kafka, миттєво знаходить патерни (наприклад, фіксація ДТП + утворення затору) та відправляє Webhook-тригери у служби реагування.
- **Пакетна обробка та зберігання (Batch Layer):**
    - **HDFS (Data Lake):** Сховище для довгострокового зберігання "холодних" сирих даних (у стислому форматі Parquet) для майбутнього аналізу та прогнозування.
    - **Apache Spark:** Виконує важкі обчислення за розкладом (наприклад, щоночі). Агрегує терабайти історичних даних з HDFS для формування статистичних зведень для мерії.
- **Аналітика (Serving Layer) — Apache Cassandra:**
    - *Роль:* Швидка розподілена NoSQL база даних, оптимізована для читання. Flink оновлює тут live-метрики (поточний трафік), а Spark записує готові зведені звіти. Міські дашборди читають дані безпосередньо звідси.
- **Інтерфейс для служб реагування:**
    - *Роль:* REST API (Webhooks), які викликаються безпосередньо з Apache Flink для миттєвої передачі критичних інцидентів (виклик поліції, швидкої чи екологічної служби).

---

### 1.2. Механізми масштабування та відмовостійкості

1. **Партиціонування (Partitioning):** Топіки в Kafka розбиті на партиції. Це дозволяє здійснювати горизонтальне масштабування — одночасно читати потік даних сотнями паралельних вузлів (TaskManagers) у Flink та Spark.
2. **Реплікація (Replication):** Kafka-топіки та блоки в HDFS мають фактор реплікації (зазвичай = 3). Якщо один фізичний сервер виходить з ладу, система миттєво перемикається на копії даних без їх втрати.
3. **Контрольні точки (Checkpointing):** `Apache Flink` періодично робить асинхронні знімки свого внутрішнього стану (State) та зберігає їх у HDFS. У разі падіння обчислювального вузла, система піднімає резервний контейнер і відновлює обробку з останнього чекпоінту, гарантуючи стійкість до збоїв.

Ми розділяємо потоки даних на два незалежні шляхи. Це гарантує, що важкі аналітичні запити (Spark) ніколи не заблокують критичний процес виявлення аварій (Flink).

**Детальна архітектурна схема системи ctOSmartCity (Lambda Architecture):**

```mermaid
flowchart TD
    subgraph IoT ["Джерела&nbsp;подій&nbsp;(IoT&nbsp;Devices)"]
        S1(["Камери<br/>Трафіку"])
        S2(["Сенсори<br/>Повітря CO2"])
        S3(["Розумні<br/>Світлофори"])
    end

    Gateway{{"IoT Gateway / MQTT"}}
    IoT -->|Стрім подій| Gateway

    Kafka[["Apache Kafka<br/>(Event Bus / Buffer)"]]
    Gateway -->|Raw JSON| Kafka

    subgraph SpeedLayer ["Speed Layer (Real-Time)"]
        Flink["Apache Flink<br/>(Потокова обробка)"]
    end

    subgraph BatchLayer ["Batch Layer (Cold Data)"]
        HDFS[("HDFS / AWS S3<br/>Data Lake")]
        Spark["Apache Spark<br/>(Пакетна обробка)"]
    end

    Kafka -->|Миттєве читання| Flink
    Kafka -.->|Скидання логів| HDFS
    HDFS -->|Щогодинні Job-и| Spark

    DB[("Apache Cassandra<br/>Serving DB")]

    Spark -->|Агреговані звіти| DB
    Flink -->|Оновлення метрик| DB

    Police{{"Екстрені Служби<br/>(API Webhooks)"}}
    Flink -->|Exactly-Once Alert| Police

    Dash(["Дашборди ctOSmartCity /<br/>Мобільний Застосунок"])
    DB -->|GraphQL / REST| Dash

    classDef external fill:#ffebee,stroke:#c62828,stroke-width:2px,color:#000;
    classDef stream fill:#e8f5e9,stroke:#2e7d32,stroke-width:2px,color:#000;
    classDef batch fill:#e3f2fd,stroke:#1565c0,stroke-width:2px,color:#000;

    class Police external;
    class Flink stream;
    class Spark,HDFS batch;
```

---

### 1.3. Обґрунтування архітектури: Оцінка навантаження (Capacity Planning)

Для підтвердження обраного технологічного стеку проведемо базові розрахунки (Back-of-the-envelope calculations) для середнього мегаполіса. 

**Вихідні дані (Assumptions):**
- Кількість IoT-сенсорів у місті: **100 000** пристроїв.
- Частота генерації подій: **1 подія/секунду** від кожного сенсора.
- Середній розмір одного повідомлення (JSON payload): **500 Bytes**.

**1. Пропускна здатність для Real-time обробки (Вимога 1):**
- **Throughput (RPS):** `100 000 events/sec`
- **Bandwidth (Вхідний трафік):** `100 000 * 500 B = 50 MB/sec`
- **Добовий трафік:** `50 MB/sec * 3600 sec * 24 h ≈ 4.3 TB/day`
> *Висновок:* Звичайні реляційні БД (PostgreSQL) не витримають безперервного потоку запису 100 тис. рядків на секунду. Саме тому на вході (Ingestion) стоїть **Apache Kafka**, здатна обробляти мільйони повідомлень за секунду завдяки послідовному запису на диск (Sequential I/O).

**2-3. Обсяг зберігання для довгострокової історії (Вимоги 2 та 3):**
- **Сирі дані за рік:** `4.3 TB/day * 365 days ≈ 1.5 PB/year`
- **З урахуванням реплікації (HDFS Replication Factor = 3):** `1.5 PB * 3 = 4.5 PB/year`
- **Оптимізація зберігання:** Оскільки ми використовуємо пакетну обробку (Batch Layer), сирі JSON-дані стискаються та конвертуються у колонковий формат **Apache Parquet** (алгоритм Snappy), що дає економію місця близько 70-80%
- **Реальний обсяг дисків:** `4.5 PB * 0.3 (30%) ≈ 1.35 PB/year`
> *Висновок:* Зберігати 1.35 Петабайта даних на рік у звичайній SQL-базі економічно неможливо. Тому для історичного аналізу використовується **HDFS** (дешеві HDD диски) та **Apache Spark**, який вміє розподілено читати Parquet-файли, не завантажуючи їх цілком в оперативну пам'ять.

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **2. Гарантії обробки (Delivery Semantics)**

У масштабах розумного міста (мільйони подій на секунду) забезпечення найвищого рівня гарантій (Exactly Once) для всього потоку даних є економічно та технічно недоцільним через високі затримки (latency) та навантаження на мережу. Тому система розмежовує семантику доставки залежно від бізнес-цінності конкретного типу даних.

### 2.1. Транспортування загальної телеметрії (At Least Once / At Most Once)

- **Сценарій:** Збір некритичних показників (температура, вологість, рівень шуму, заповненість паркінгів), їх буферизація у Kafka, збереження та подальша пакетна обробка.
- **Базові гарантії:** Залежно від етапу життєвого циклу застосовується `At Least Once` (Щонайменше один раз) або `At Most Once` (Щонайбільше один раз).
- **Обґрунтування та логіка по етапах:**
    - **Транспортування (Ingestion):** Від сенсорів до Kafka використовується `At Most Once` заради максимальної пропускної здатності. Втрата одного пакету з показником 22°C серед тисяч інших не вплине на загальну аналітику району.
    - **Обробка потоків (Stream Processing):** Для загальної телеметрії Flink працює в режимі `At Least Once` (без важких двофазних комітів), щоб мінімізувати затримки (latency).
    - **Збереження (Storage):** Під час скидання сирих логів з Kafka у Data Lake (HDFS) гарантується `At Least Once`, оскільки збереження повної історії подій є пріоритетом. Під час мережевих ретраїв тут неминуче виникатимуть дублікати.
    - **Формування звітів (Reporting):** На етапі агрегації (Spark) та запису результатів у Serving Layer (Cassandra) ми нівелюємо всі накопичені дублікати через ідемпотентний запис (Upsert) за композитним ключем:
      > **Формула Primary Key (Cassandra):**
      > `PK = PartitionKey(sensor_id) + ClusteringKey(timestamp)`
      > `UPDATE telemetry SET value = X WHERE sensor_id = Y AND timestamp = Z`
      > *(Завдяки такій структурі будь-який дубльований запис просто перезапише сам себе без створення аномалій).*

---

### 2.2. Критичні події: Аварії та Тривоги (Exactly Once)

- **Сценарій:** Виявлення ДТП камерами трафіку, фіксація критичного перевищення CO₂, автоматичний виклик швидкої або поліції
- **Гарантія:** Суворе `End-to-End Exactly Once` (Точно один раз)
- **Обґрунтування:** Втрата події означає відсутність реакції на аварію (неприпустимий ризик для життя). Дублювання події через мережевий збій означає відправку кількох екіпажів поліції на один виклик (марнотратство муніципальних ресурсів).

---

### 2.3. Забезпечення Exactly Once семантики для критичних підсистем

Реалізація наскрізного (End-to-End) Exactly Once для автоматичного реагування на аварії досягається комбінацією трьох етапів життєвого циклу даних:

1. **Читання (Source):** Apache Flink не покладається на авто-коміти Kafka. Він зберігає поточні зміщення (Kafka Offsets) у власному розподіленому стані (State). У разі збою Flink точно знає, з якого місця потрібно відновити читання, виключаючи втрату даних.
2. **Обробка (Processing):** Використовується алгоритм *Chandy-Lamport* для створення узгоджених знімків стану (Distributed Snapshots). Якщо вузол падає під час обчислення "вікна" подій, система відновлюється з останнього чекпоінту без дублювання внутрішніх обчислень.
3. **Запис та надсилання сповіщень (Sink):** На фінальному етапі застосовуються два ключові патерни:
    - **Flink Two-Phase Commit (2PC):** Забезпечує наскрізне Exactly Once при роботі з транзакційними системами. Зміни комітяться в зовнішню систему лише тоді, коли Flink успішно завершив Checkpoint.
    - **Ідемпотентність API (Idempotency Keys):** При виклику зовнішніх REST API Flink генерує унікальний криптографічний ідентифікатор інциденту.
      > **Формула Idempotency Key:**
      > `Idempotency_Key = SHA-256(camera_id || timestamp || event_type)`
      > *(Конкатенація ідентифікатора джерела, мілісекундної мітки часу та типу події з подальшим хешуванням).*

**Схема обробки критичної події (Exactly-Once Transaction Flow):**

```mermaid
sequenceDiagram
    autonumber
    participant K as Apache Kafka
    participant F as Apache Flink
    participant H as HDFS (State)
    participant API as Police REST API

    K->>F: Читання стріму подій (Offset: N)
    F->>F: Complex Event Processing (виявлено ДТП)
    F->>F: Генерація Idempotency_Key (SHA-256)

    Note over K,API: --- Фаза 1: Pre-Commit (Chandy-Lamport) ---
    F->>H: Асинхронний Checkpoint (State + Offset N)
    H-->>F: Checkpoint ACK (Успішно)

    Note over K,API: --- Фаза 2: Зовнішній виклик та Commit ---
    F->>API: POST /alert (Header: Idempotency-Key)

    alt Мережевий збій (Тайм-аут)
        API-->>F: Timeout Exception
        Note over F,API: Flink робить Retry через збій мережі
        F->>API: RETRY POST /alert (той самий Idempotency-Key)
        API-->>F: HTTP 200 OK (Дублікат відхилено базою)
    else Успішна доставка з першого разу
        API-->>F: HTTP 201 Created
    end

    F->>K: Фіксація зміщення (Commit Offset N)
```

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **3. Відновлення після збоїв (Fault Tolerance та High Availability)**

Система спроектована з урахуванням аксіоми: *Апаратні збої неминучі*. Оскільки платформа складається з тисяч серверів, вихід з ладу окремих вузлів є штатною ситуацією, яка обробляється автоматично.

### 3.1. Вихід з ладу брокера Kafka (Ingestion Layer)
- **Механізм:** Топіки Kafka налаштовані з `Replication Factor = 3` та `min.insync.replicas = 2`.
- **Відновлення:** Якщо один сервер (Broker) фізично "згорає", кластер автоматично обирає нового лідера для партицій з групи ISR (In-Sync Replicas) за лічені мілісекунди. IoT-шлюзи продовжують писати дані без втрат.

---

### 3.2. Падіння вузла потокової обробки (Apache Flink)
- **Механізм:** Flink використовує механізм **Checkpointing** на базі алгоритму *Chandy-Lamport*. Кожні 10 секунд він робить асинхронний знімок свого внутрішнього стану (State) та поточного зміщення (Offset) у Kafka, зберігаючи цей знімок у розподілену файлову систему (HDFS).
- **Відновлення:** Якщо вузол (TaskManager) падає через нестачу пам'яті (OOM) або збій заліза, JobManager піднімає новий контейнер, завантажує останній Checkpoint з HDFS, "відмотує" Kafka на 10 секунд назад і переграє події. Це гарантує відсутність втрат стану вікон обчислення.

---

### 3.3. Збій під час пакетної обробки (Apache Spark)
- **Механізм:** Замість дорогої реплікації проміжних даних у RAM, Spark використовує концепцію **RDD Lineage (Граф походження даних)**.
- **Відновлення:** Якщо Worker-вузол падає під час важкої нічної генерації звітів, Spark Driver просто перепризначає втрачені завдання (Tasks) на інші здорові вузли. Вони наново вичитують необхідні блоки з HDFS і переобчислюють лише втрачену частину даних.

---

### 3.4. Збій сховищ даних (HDFS та Apache Cassandra)
- **Збій Data Lake (HDFS):**
  - **Механізм:** HDFS автоматично розбиває файли на блоки (Block Size = 128 MB) і зберігає 3 копії кожного блоку на різних фізичних серверах (DataNodes), бажано в різних стійках (**Rack Awareness**).
  - **Відновлення:** При збої диска або сервера, NameNode виявляє відсутність heartbeat-сигналу і автоматично ініціює копіювання втрачених блоків з вцілілих серверів на нові, підтримуючи фактор реплікації.
- **Стійкість аналітичної БД (Cassandra):**
  - **Механізм:** База має Masterless (Peer-to-Peer) архітектуру без єдиної точки відмови.
  - **Відновлення:** Якщо вузол падає під час запису, сусідні вузли зберігають ці дані тимчасово у себе (механізм **Hinted Handoff**). Щойно вузол повертається в стрій, сусіди автоматично "віддають" йому пропущені дані.

**Схема процесу відновлення потокової обробки (Apache Flink State Recovery):**

```mermaid
sequenceDiagram
    autonumber
    participant TM as Flink TaskManager (Worker)
    participant JM as Flink JobManager (Master)
    participant H as HDFS (State Backend)
    participant K as Apache Kafka

    TM--xJM: Втрата Heartbeat (OOM / Апаратний збій)
    Note over JM: Детектування збою кластера
    JM->>JM: Зупинка поточного графа обчислень

    Note over JM,H: Ініціалізація відновлення
    JM->>H: Запит останнього успішного Checkpoint
    H-->>JM: Метадані стану (RocksDB) + Kafka Offsets

    JM->>TM: Деплой графа на новий/резервний TaskManager
    TM->>H: Завантаження фізичного стану в пам'ять

    Note over TM,K: Відновлення консистентності
    TM->>K: Запит на читання стріму (відмотування Offset)
    K-->>TM: Відновлення потоку даних
    Note over TM,K: Обробка продовжується без втрати подій (Exactly-Once)
```

---

### 3.5. Контроль цілісності даних (Data Integrity)
Система не лише відновлює дані після збоїв, але й постійно перевіряє їх на пошкодження (bit rot) на фізичному та логічному рівнях:
- **Фізична цілісність (HDFS та Kafka):** Під час кожного запису розраховуються контрольні суми (CRC32). При читанні вузол знову обчислює хеш і звіряє його. Якщо диск пошкодив дані, контрольна сума не співпаде. Система миттєво відкине цей блок і прозоро для клієнта завантажить здорову копію з іншої репліки (DataNode або ISR брокера).
- **Логічна цілісність (Spark):** Базується на незмінності (Immutability) даних. RDD у Spark не можна змінити після створення. Це гарантує, що переобчислення втраченого завдання завжди дасть математично ідентичний результат без побічних ефектів чи дублювань.

---

### 3.6. Катастрофостійкість (Disaster Recovery): Війна та стихійні лиха

Локальна надмірність (наприклад, Rack Awareness) не захищає систему від повного фізичного знищення або блекауту всього дата-центру (через бойові дії, пожежу, повінь). Для забезпечення безперервної роботи критичної інфраструктури міста застосовується стратегія **Geo-Redundancy (Мультирегіональна архітектура)**:

- **Топологія:** Платформа розгорнута у двох географічно віддалених дата-центрах (наприклад, Основний ЦОД — Київ, Резервний — хмарний регіон AWS `eu-central-1` у Франкфурті) за моделлю *Active-Passive* або *Active-Active*.
- **Гео-реплікація транспорту (Kafka):** Використовується інструмент **MirrorMaker 2** для безперервної асинхронної крос-кластерної реплікації критичних топіків (аварії, виклики швидкої) з основного дата-центру в резервний.
- **Глобальна аналітика (Cassandra):** Завдяки нативній підтримці мульти-ЦОД архітектури (`NetworkTopologyStrategy`), Cassandra автоматично підтримує узгоджені копії даних у обох регіонах. У разі втрати основного ЦОД, мобільні застосунки містян та дашборди мерії автоматично перемикаються на резервний через DNS-маршрутизацію.
- **Резервування Data Lake:** Історичні дані та чекпоінти Flink безперервно копіюються у віддалений регіон за допомогою асинхронної крос-регіональної реплікації (Cross-Region Replication для AWS S3 або DistCp для HDFS).
- **Метрики стійкості (RTO та RPO):** Recovery Time Objective (час перемикання екстрених служб на резервний ЦОД) складає лічені хвилини. Recovery Point Objective (обсяг безповоротно втрачених даних у момент фізичного знищення основного ЦОД) зводиться до мілісекунд завдяки безперервній потоковій гео-реплікації.

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **4. Випадки використання та SQL-запити**

Цей розділ демонструє практичне застосування Лямбда-архітектури: як одна й та сама система ефективно обробляє реал-тайм тригери та важку історичну аналітику.

### 4.1. Потоковий аналітичний запит (Stream Processing / Apache Flink)
- **Завдання:** Виявлення критичного перевищення рівня CO₂ (вище 1000 ppm) на конкретному перехресті протягом останніх 5 хвилин.
- **Обґрунтування (Чому Stream?):** Реакція має бути миттєвою (наприклад, автоматична команда на IoT Gateway увімкнути зелене світло на всіх світлофорах вулиці, щоб "розсмоктати" затор або надіслати push-сповіщення у **мобільний застосунок** містян). Чекати пакетної обробки неможливо через загрозу здоров'ю.
- **Математична модель вікна (Tumbling Window):** У Flink кожна подія жорстко прив'язується до неперетинного часового інтервалу (вікна) за формулою:
  > $Window_{start} = \lfloor \frac{Event_{time}}{Window_{size}} \rfloor \times Window_{size}$
  > $Window_{end} = Window_{start} + Window_{size}$
  *(Завдяки цьому події, що запізнилися через мережу (Late Data), гарантовано потраплять у правильне історичне вікно, а не в поточне).*

```sql
-- Flink SQL: Безперервна агрегація потоку з Kafka у реальному часі
SELECT 
    sensor_id,
    TUMBLE_END(event_time, INTERVAL '5' MINUTE) AS alert_time,
    AVG(co2_level) AS avg_co2
FROM city_sensors
GROUP BY 
    TUMBLE(event_time, INTERVAL '5' MINUTE), 
    sensor_id
HAVING AVG(co2_level) > 1000;
```

---

### 4.2. Пакетний аналітичний запит (Batch Processing / Apache Spark)
- **Завдання:** Підрахунок середнього та пікового рівня шуму по районах міста за останній звітний місяць для департаменту урбаністики.
- **Обґрунтування (Чому Batch?):** Запит сканує терабайти історичних даних за 30 днів. Виконувати це в in-memory стрімінгу технічно недоцільно і надто дорого. Spark ідеально підходить для важких `GROUP BY` операцій над "холодними" даними.
- **Оптимізація виконання (Predicate Pushdown):** Завдяки колонковому формату зберігання (Parquet в HDFS), фільтр `WHERE event_time` передається на рівень файлової системи. Spark фізично зчитує лише файли потрібного місяця, ігноруючи решту петабайтів даних (Zero-copy read).

**Схема фізичного плану виконання запиту (Spark Execution DAG):**

```mermaid
graph LR
    A[(HDFS: Parquet<br/>Data Lake)] -->|1. Predicate Pushdown <br/> Фільтр по 'month'| B(Map/Project<br/>Вибірка колонок)
    B -->|2. Exchange/Shuffle <br/> Мережевий&nbsp;обмін&nbsp;по&nbsp;'district'| C(HashAggregatet<br/>AVG, MAX)
    C -->|3. Sort <br/> Сортування DESC| D(Result Set<br/>Презентаційний шар)

    style A fill:#f9f,stroke:#333,stroke-width:2px
    style C fill:#bbf,stroke:#333,stroke-width:2px
```

```sql
-- Spark SQL: Пакетна обробка історичних даних з Data Lake
SELECT 
    district_name,
    DATE_TRUNC('month', event_time) AS report_month,
    AVG(noise_db) AS avg_noise_level,
    MAX(noise_db) AS peak_noise_level
FROM hdfs_city_telemetry_history
WHERE event_time >= '2026-02-01' AND event_time < '2026-03-01'
GROUP BY 
    district_name, 
    DATE_TRUNC('month', event_time)
ORDER BY avg_noise_level DESC;
```

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **5. Захист даних та конфіденційність (Security & Privacy)**

Платформа **ctOSmartCity** обробляє величезні масиви чутливих даних (переміщення громадян, відеопотоки, геолокація). Для запобігання витокам та забезпечення відповідності суворим стандартам конфіденційності (наприклад, GDPR) архітектура системи будується на принципах **Privacy by Design** та **Zero Trust** (Нульової довіри).

### 5.1. Анонімізація та маскування чутливих даних (Data Masking)
Найбільший ризик конфіденційності несуть камери трафіку, які фіксують обличчя пішоходів та номерні знаки автомобілів. Сирі медіадані ніколи не повинні потрапляти до центрального Data Lake.
- **Обробка на периферії (Edge Computing):** Виявлення ДТП та розпізнавання об'єктів відбувається безпосередньо на IoT-шлюзах (Edge Devices) за допомогою легковагових ML-моделей.
- **Незворотне хешування (Salted Hashing):** Замість передачі реального номера авто, система генерує криптографічний хеш із використанням динамічної "солі".
  > **Формула анонімізації події:**
  > `Event = { "speed": 85, "plate_hash": SHA-256(plate_number + DAILY_SALT), "location": "Sector_7" }`
  > *(Це дозволяє Spark агрегувати статистику унікальних автомобілів у районі без фактичного знання того, чиї це автомобілі. Сіль оновлюється щодоби, унеможливлюючи довгострокове стеження).*

---

### 5.2. Шифрування даних (Encryption In Transit & At Rest)

Навіть анонімізовані дані потребують криптографічного захисту від перехоплення та крадіжки фізичних носіїв.

- **In Transit (У русі):** Будь-яка комунікація між компонентами (MQTT від сенсорів, передача з Kafka у Flink, виклики REST API) відбувається виключно через **TLS 1.3**. Внутрішній трафік кластера захищається за допомогою mTLS (Mutual TLS), де кожен вузол підтверджує свою автентичність сертифікатом.
- **At Rest (У стані спокою):** На рівні HDFS та Cassandra увімкнено прозоре шифрування (TDE).
- **Управління ключами (KMS):** Використовується патерн **Envelope Encryption** (Конвертне шифрування) через AWS KMS / HashiCorp Vault з ротацією кожні 90 днів. Майстер-ключ ніколи не покидає захищене сховище.
  > **Логіка конвертного шифрування:**
  > `Ciphertext = AES_256_Encrypt(Plaintext, Data_Key)`
  > `Encrypted_Data_Key = RSA_Encrypt(Data_Key, Master_Key)`
  > *(На диск зберігається лише `Ciphertext` разом із зашифрованим `Encrypted_Data_Key`).*

---

### 5.3. Управління доступом, авторизація та безпека кешу (IAM & Cache Security)

Система суворо розмежовує права доступу для різних споживачів даних (поліція, екологічна служба, звичайні громадяни).

- **Role-Based Access Control (RBAC):** Доступ до аналітичної бази (Cassandra) та топіків Kafka регулюється через політики RBAC. Наприклад, сервіс екології має доступ лише до топіка `metrics.air_quality`.
- **Автентифікація клієнтів:** Міські дашборди та **мобільний застосунок** для мешканців взаємодіють із системою через API Gateway. Автентифікація реалізована за протоколом OAuth 2.0 / OpenID Connect (через Keycloak).
- **Короткострокові токени:** Мобільний застосунок використовує JWT (JSON Web Tokens) із коротким часом життя (TTL = 15 хвилин), що мінімізує ризики у разі компрометації пристрою.
- **Запобігання витокам через кеш (Cache Security & IDOR Prevention):** Щоб унеможливити ситуацію, коли API Gateway помилково віддає чутливі дані одного користувача іншому через агресивне кешування, система суворо розділяє потоки:
  - Публічні дані (загальні затори) кешуються глобально.
  - Персоналізовані запити завжди маркуються HTTP-заголовками `Cache-Control: private, no-store` та `Vary: Authorization`. Додатково, бекенд перевіряє прив'язку ресурсу до конкретного користувача (захист від **Insecure Direct Object Reference**), гарантуючи, що користувач може отримати лише ті дані, які відповідають ідентифікатору (`sub`) у його власному JWT.

---

### 5.4. Аудит та відповідність (Audit Trails)

Для розслідування можливих інцидентів безпеки всі дії адміністраторів та операторів системи логуються.

- **WORM-сховище:** Журнали аудиту (Audit Logs) записуються в окреме сховище з політикою **Write-Once-Read-Many (WORM)**, що математично унеможливлює видалення або зміну записів навіть користувачами з правами `root`. Це є критичною вимогою для проходження сертифікації безпеки (ISO 27001).

**Схема безпечного доступу до даних (Zero Trust Auth Flow):**

```mermaid
sequenceDiagram
    autonumber
    participant App as Мобільний застосунок
    participant IDP as IAM (Keycloak)
    participant GW as API Gateway (Cache Layer)
    participant DB as Cassandra (Serving Layer)

    App->>IDP: POST /auth (Біометрія / Пароль)
    Note over IDP: Валідація облікових даних
    IDP-->>App: Повернення JWT Access Token (TTL=15m)

    App->>GW: GET /user/routes/history (Header: Bearer JWT)

    Note over GW: Перевірка підпису JWT (JWKS)
    GW->>GW: Інспекція кешу (Cache-Control: private)

    alt Доступ дозволено (Перевірка IDOR успішна)
        GW->>DB: Запит маршрутів для User_ID з токена
        DB-->>GW: Результат (JSON)
        GW-->>App: HTTP 200 OK (Cache-Control: no-store)
    else Спроба підміни User_ID (IDOR Attack)
        App->>GW: GET /user/999/routes (чужий ID)
        GW-->>App: HTTP 403 Forbidden (Token "sub" mismatch)
    end
```

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **6. Практична симуляція ctOSmartCity: Виявлення ДТП у реальному часі (Proof of Concept)**

Щоб продемонструвати логіку потокової обробки (Speed Layer) та підтвердити валідність Flink SQL запиту з Розділу 4.1, ми розробили синтетичну симуляцію поведінки системи на перехресті.

**Сценарій симуляції:**
1. **Нормальний стан (08:00 - 08:14):** Трафік рухається вільно (~50 км/год), рівень CO₂ в межах екологічної норми (~400 ppm).
2. **Аварія / Аномалія (08:15 - 08:40):** Стається ДТП. Швидкість потоку падає до критичних 5 км/год. Через утворення затору рівень CO₂ різко пробиває поріг у 1000 ppm.
3. **Реакція системи:** Алгоритм потокової аналітики фіксує перетин порогу та генерує `Critical Alert` для автоматичного перемикання світлофорів.

In [22]:
# Симуляція даних - Створення анімації Plotly
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import base64
from IPython.display import HTML, display
import warnings

warnings.filterwarnings('ignore')

def simulate_ctos_sensors(minutes: int = 60, accident_start: int = 15, accident_end: int = 40) -> pd.DataFrame:
    np.random.seed(42)
    time_index = pd.date_range(start="2026-03-12 08:00", periods=minutes, freq="1min")

    speed_data, co2_data, alerts = [], [], []

    for i in range(minutes):
        if accident_start <= i <= accident_end:
            speed = max(0, np.random.normal(5, 2))  
            co2 = np.random.normal(1200, 150)       
        else:
            speed = min(80, np.random.normal(50, 5))
            co2 = np.random.normal(400, 20)

        speed_data.append(speed)
        co2_data.append(co2)

        if co2 > 1000 and speed < 15:
            alerts.append(1200)
        else:
            alerts.append(None)

    return pd.DataFrame({
        'Time': time_index,
        'AvgSpeed': speed_data,
        'CO2_Level': co2_data,
        'AlertTrigger': alerts
    })

df = simulate_ctos_sensors()

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(x=df['Time'][:1], y=df['CO2_Level'][:1], name="Рівень CO₂ (ppm)", 
                         line=dict(color='#ff3333', width=2), fill='tozeroy', fillcolor='rgba(255, 51, 51, 0.2)'), secondary_y=False)
fig.add_trace(go.Scatter(x=df['Time'][:1], y=df['AvgSpeed'][:1], name="Середня швидкість (км/год)", 
                         line=dict(color='#00ffcc', width=3)), secondary_y=True)
fig.add_trace(go.Scatter(x=df['Time'][:1], y=df['AlertTrigger'][:1], mode='markers', name="⚠️ ctOS ALERT",
                         marker=dict(color='yellow', size=12, symbol='triangle-up', line=dict(color='black', width=1))), secondary_y=False)

frames = []
for i in range(1, len(df) + 1):
    frames.append(go.Frame(data=[
        go.Scatter(x=df['Time'][:i], y=df['CO2_Level'][:i]),
        go.Scatter(x=df['Time'][:i], y=df['AvgSpeed'][:i]),
        go.Scatter(x=df['Time'][:i], y=df['AlertTrigger'][:i])
    ], name=str(i)))
fig.frames = frames

animation_time_second = 15
frame_duration = int((animation_time_second * 1000) / len(df))

sliders = [dict(
    steps=[dict(method='animate', 
                args=[[str(i)], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))],
                label=df['Time'].dt.strftime('%H:%M').iloc[i-1]) for i in range(1, len(df) + 1)],
    active=0,
    transition=dict(duration=0),
    x=0, y=-0.1,
    currentvalue=dict(font=dict(size=14, color="orange"), prefix="Час: ", visible=True, xanchor="right"),
    len=1.0
)]

fig.update_layout(
    title="<b>ctOSmartCity Dashboard:</b> Симуляція моніторингу перехрестя (Виявлення ДТП)",
    template="plotly_dark",
    hovermode="x unified",
    hoverlabel=dict(namelength=-1), 
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    xaxis=dict(range=[df['Time'].min(), df['Time'].max()], title="Час"),
    yaxis=dict(range=[0, 1600], title="Рівень CO₂ (ppm)"),
    yaxis2=dict(range=[0, 100], title="Швидкість потоку (км/год)", showgrid=False),
    sliders=sliders,
    updatemenus=[dict(
        type="buttons",
        showactive=False,
        x=0.02, y=1.15,
        xanchor="left", yanchor="top",
        direction="right",
        pad={"r": 15, "t": 25},
        buttons=[
            dict(label=f"▶ Старт ({animation_time_second} сек)", method="animate",
                 args=[None, {"frame": {"duration": frame_duration, "redraw": True}, "fromcurrent": True}]),
            dict(label="⏸ Пауза", method="animate",
                 args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}])
        ]
    )]
)

fig.add_hline(y=1000, line_dash="dash", line_color="orange", 
              annotation_text="Критичний поріг CO₂ (1000 ppm)", 
              annotation_position="top left", secondary_y=False)

plot_html = fig.to_html(include_plotlyjs='cdn', full_html=True)
b64_chart = base64.b64encode(plot_html.encode()).decode()

download_link = f'<div style="text-align: center; margin-bottom: 10px;"><a href="data:text/html;base64,{b64_chart}" download="ctos_dashboard.html" style="color:rgb(255, 51, 52); text-decoration: none; font-size: 16px; font-weight: bold; padding: 8px 16px; border: 1px solid rgb(255, 51, 52); border-radius: 5px; background-color: rgba(255, 51, 52, 0.1);">🚗 Завантажити інтерактивний ctOSmartCity</a></div>'
iframe_display = f'<iframe src="data:text/html;base64,{b64_chart}" width="100%" height="650px" frameborder="0"></iframe>'

display(HTML(download_link))
display(HTML(iframe_display))

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **Загальний висновок (Executive Summary)**

Спроєктована архітектура муніципальної системи **ctOSmartCity** демонструє зріле розуміння патернів обробки великих даних (Big Data). Впровадження Лямбда-архітектури дозволило створити відмовостійку, масштабовану платформу, яка досягає чотирьох головних архітектурних цілей:

1. **Розділення обов'язків (Separation of Concerns):** Відокремлення швидкого контуру (Speed Layer: Kafka + Flink) від холодного сховища (Batch Layer: HDFS + Spark) дозволяє безперебійно обробляти петабайти даних. Це гарантує, що побудова важких місячних звітів для мерії ніколи не заблокує ресурси, необхідні для миттєвої реакції на екстрені події.
2. **Оптимізація ресурсів та гарантії (Delivery Guarantees):** Усвідомлений архітектурний компроміс (trade-off) між продуктивністю та надійністю. Використання `At Least Once` для базової телеметрії економить ресурси, тоді як `Exactly Once` (через двофазний коміт 2PC) для критичних тригерів повністю виключає ризик дублювання викликів швидкої допомоги під час ДТП.
3. **Абсолютна відмовостійкість (Resiliency & Zero Data Loss):** Вбудовані механізми реплікації (Geo-Redundancy) та Checkpointing гарантують, що навіть у разі падіння цілого дата-центру (наприклад, через блекаут), система не "завмре" і відновить роботу з точного місця (offset) без втрати цілісності.
4. **Захист приватності містян (Privacy by Design):** Завдяки Edge Computing, конвертному шифруванню та суворому захисту кешу (IDOR Prevention), особисті дані громадян (як-от маршрути у **мобільному застосунку**) надійно захищені від перехоплення та витоків.

Запропонована архітектура повністю покриває вимоги сучасного Smart City, ідеально балансуючи між блискавичною швидкістю реакції, математичною точністю та абсолютною безпекою. Цей HLD-документ (High-Level Design) є повністю життєздатним і готовим до переходу на фазу низькорівневого проєктування (LLD) та імплементації командами розробки.